# Video Quality Enhancement with Real-ESRGAN

This notebook upscales and enhances video quality using Real-ESRGAN.

**IMPORTANT:** Switch to GPU runtime before running:
- Go to Runtime > Change runtime type
- Select T4 GPU
- Click Save

**Steps:**
1. Install Real-ESRGAN
2. Upload your video
3. Enhance video quality (upscale 2x or 4x)
4. Display and download the enhanced result

## Cell 1: Setup - Install Real-ESRGAN

In [ ]:
import os
import torch

# Check GPU
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

# Clone Real-ESRGAN
if not os.path.exists('/content/Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN
    %cd /content/Real-ESRGAN
    !pip install -q basicsr facexlib gfpgan
    !pip install -q -r requirements.txt
    !python setup.py develop --quiet
    print('Real-ESRGAN installed successfully!')
else:
    %cd /content/Real-ESRGAN
    print('Real-ESRGAN already installed.')

# Download models
model_dir = '/content/Real-ESRGAN/weights'
os.makedirs(model_dir, exist_ok=True)

model_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth'
model_path = f'{model_dir}/realesr-animevideov3.pth'
if not os.path.exists(model_path):
    !wget -q {model_url} -O {model_path}
    print('Video enhancement model downloaded.')

# General model for realistic content
general_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
general_path = f'{model_dir}/RealESRGAN_x4plus.pth'
if not os.path.exists(general_path):
    !wget -q {general_url} -O {general_path}
    print('General enhancement model downloaded.')

print('\nSetup complete! Ready to enhance videos.')

## Cell 2: Upload Video

In [ ]:
from google.colab import files as colab_files
from pathlib import Path

# Configuration
UPSCALE_FACTOR = 2  # 2 or 4 (4x takes much longer)
MODEL_TYPE = 'general'  # 'general' for realistic, 'anime' for animated content

# Upload video
input_dir = Path('/content/enhance_input')
input_dir.mkdir(exist_ok=True)
output_dir = Path('/content/enhance_output')
output_dir.mkdir(exist_ok=True)

print(f'Settings: {UPSCALE_FACTOR}x upscale, {MODEL_TYPE} model')
print()
print('Upload your video file (mp4 recommended):')
uploaded = colab_files.upload()

video_filename = list(uploaded.keys())[0]
input_video = input_dir / video_filename
with open(input_video, 'wb') as f:
    f.write(uploaded[video_filename])

file_size = input_video.stat().st_size / (1024 * 1024)
print(f'\nUploaded: {video_filename} ({file_size:.1f} MB)')
print('Ready to enhance!')

## Cell 3: Enhance Video Quality

In [ ]:
import subprocess
import time

# Select model based on content type
if MODEL_TYPE == 'anime':
    model_name = 'realesr-animevideov3'
    suffix = 'anime'
else:
    model_name = 'RealESRGAN_x4plus'
    suffix = 'enhanced'

output_video = output_dir / f'{input_video.stem}_{suffix}_{UPSCALE_FACTOR}x.mp4'

# First, extract frames
frames_dir = Path('/content/frames_input')
enhanced_frames_dir = Path('/content/frames_output')
frames_dir.mkdir(exist_ok=True)
enhanced_frames_dir.mkdir(exist_ok=True)

print('Step 1/4: Extracting frames from video...')
!ffmpeg -y -i "{input_video}" -qscale:v 1 -qmin 1 -qmax 1 -vsync 0 "{frames_dir}/frame_%08d.png" -loglevel warning

frame_count = len(list(frames_dir.glob('*.png')))
print(f'  Extracted {frame_count} frames.')

print(f'\nStep 2/4: Enhancing frames with Real-ESRGAN ({UPSCALE_FACTOR}x)...')
print('  This may take a while depending on video length and resolution...')
start_time = time.time()

!python /content/Real-ESRGAN/inference_realesrgan.py \
    -n {model_name} \
    -i "{frames_dir}" \
    -o "{enhanced_frames_dir}" \
    -s {UPSCALE_FACTOR} \
    --suffix "" \
    --ext png

elapsed = time.time() - start_time
print(f'  Enhancement complete! ({elapsed:.1f}s)')

# Get original video FPS
print('\nStep 3/4: Getting video properties...')
fps_result = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-select_streams', 'v:0',
     '-show_entries', 'stream=r_frame_rate', '-of', 'csv=p=0',
     str(input_video)],
    capture_output=True, text=True
)
fps_str = fps_result.stdout.strip()
if '/' in fps_str:
    num, den = fps_str.split('/')
    fps = float(num) / float(den)
else:
    fps = float(fps_str) if fps_str else 30.0
print(f'  Original FPS: {fps}')

# Reassemble video
print('\nStep 4/4: Assembling enhanced video...')
!ffmpeg -y -framerate {fps} -i "{enhanced_frames_dir}/frame_%08d.png" \
    -i "{input_video}" -map 0:v -map 1:a? \
    -c:v libx264 -crf 17 -pix_fmt yuv420p -c:a copy \
    "{output_video}" -loglevel warning

if output_video.exists():
    out_size = output_video.stat().st_size / (1024 * 1024)
    print(f'\nEnhancement COMPLETE!')
    print(f'  Input: {file_size:.1f} MB')
    print(f'  Output: {out_size:.1f} MB ({UPSCALE_FACTOR}x upscale)')
    print(f'  Saved to: {output_video}')
else:
    print('ERROR: Output video was not created.')

# Cleanup frames to save space
!rm -rf "{frames_dir}" "{enhanced_frames_dir}"
print('Temporary frames cleaned up.')

## Cell 4: Display and Download Result

In [ ]:
from IPython.display import HTML
from base64 import b64encode
from google.colab import files as colab_files

if output_video.exists():
    # Display video (only if small enough for inline display)
    out_size_mb = output_video.stat().st_size / (1024 * 1024)
    
    if out_size_mb < 50:  # Only inline if under 50MB
        video_data = open(output_video, 'rb').read()
        video_b64 = b64encode(video_data).decode()
        display(HTML(f"""
        <h3>Enhanced Video Result ({UPSCALE_FACTOR}x Upscale)</h3>
        <video width="800" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
        <p>Size: {out_size_mb:.1f} MB</p>
        """))
    else:
        print(f'Video too large for inline display ({out_size_mb:.1f} MB).')
        print('Downloading directly...')
    
    # Download
    print(f'\nDownloading enhanced video ({out_size_mb:.1f} MB)...')
    colab_files.download(str(output_video))
    print('Download started!')
else:
    print('No enhanced video found. Please run Cell 3 first.')